# 실습 8: 기준 모델과 추천 모델
- 상황: 아무것도 안 해도 93점이 나온다는 걸 알았다
- 목표: 비교할 기준을 먼저 만들고, 그 위에서 진짜 모델을 재본다

## Step 0. 앞 실습까지 재현하기

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

sensor_cols = [c for c in df.columns if c.startswith("sensor_")]
for c in sensor_cols:
    df[c] = df[c].fillna(df[c].median())

df["불량여부"] = (df["result"] == "불량").astype(int)

X = df[sensor_cols]
y = df["불량여부"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("학습용:", X_train.shape, " 불량 건수:", y_train.sum())
print("시험용:", X_test.shape, " 불량 건수:", y_test.sum())


학습용: (1253, 50)  불량 건수: 83
시험용: (314, 50)  불량 건수: 21


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 모델을 비교할 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 기준 모델 | 학습을 전혀 하지 않고 늘 같은 답만 내놓는 모델. 비교의 바닥선이 된다 |
| 학습 | 답이 붙은 기록을 넣어 규칙을 찾게 하는 일 |
| 예측 | 처음 보는 기록에 답을 붙이는 일 |
| 정확도 | 전체 중 맞힌 비율. 오늘 쓰는 유일한 점수이고, 내일 이 점수를 의심하게 된다 |

## Step 2. 게으름뱅이 모델 만들기

In [2]:
# numpy - 숫자 묶음을 다루는 도구를 np라는 짧은 이름으로 불러온다
import numpy as np

# 시험용 개수만큼 전부 0(양품)으로 채운 답안지를 만든다. 학습은 하지 않았다
기준예측 = np.zeros(len(y_test), dtype=int)

# 맞힌 개수 ÷ 전체 개수
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델이 불량이라 한 건수:", 기준예측.sum())
print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")

기준 모델이 불량이라 한 건수: 0
기준 모델 정확도: 93.31 %


In [3]:
# 시험용에서 양품이 몇 건, 불량이 몇 건인지
print("시험용 양품:", (y_test == 0).sum(), "건")
print("시험용 불량:", (y_test == 1).sum(), "건")

# 전부 양품이라 답하면 -> 양품은 다 맞고, 불량은 다 틀린다
print("맞힌 것:", (y_test == 0).sum(), "/", len(y_test))

시험용 양품: 293 건
시험용 불량: 21 건
맞힌 것: 293 / 314


[기준 모델이 높은 점수를 받는 이유]<br>
시험용 [314]건 중 양품이 [293]건이다.<br>
전부 양품이라 답하면 [293]건은 자동으로 맞는다.<br>
불량 [21]건은 전부 놓치지만, 개수가 적어 점수에 거의 영향이 없다.

## Step 4. 모델을 추천받기

[추천받은 모델]<br>
1. [로지스틱 회귀] - [둘 중 하나를 고르는 문제의 기본이고, 어느 열이 얼마나 작용했는지 볼 수 있다]<br>
2. [의사결정나무] - [자르는 기준이 눈에 보여서 설명하기 쉽다]<br>
내가 고른 것 : [로지스틱 회귀]

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 로지스틱 회귀는 열마다 자릿수가 다르면 계수가 왜곡되므로 표준화가 필요하다
# 학습용 기준으로 평균·표준편차를 구하고(fit), 학습용·시험용 둘 다 그 기준으로 변환한다(transform)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 기본 상태 그대로 (class_weight 등 불균형 보정 설정 없음)
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

예측 = model.predict(X_test_scaled)

정확도 = accuracy_score(y_test, 예측) * 100
불량_예측_건수 = int((예측 == 1).sum())
그중_실제_불량_건수 = int(((예측 == 1) & (y_test == 1)).sum())

print("정확도:", round(정확도, 2), "%")
print("불량이라고 예측한 건수:", 불량_예측_건수)
print("그중 실제로 불량이었던 건수:", 그중_실제_불량_건수)


정확도: 93.95 %
불량이라고 예측한 건수: 2
그중 실제로 불량이었던 건수: 2


## Step 6. 모델 기록표

| 모델 | 왜 썼나 | 정확도 | 불량이라 한 건수 | 그중 진짜 |
|---|---|---|---|---|
| 기준 모델 (전부 양품) | 비교할 바닥선 | [93.31]% | [0] | [0] |
| [로지스틱 회귀] | [분류의 기본이고 결과를 설명하기 쉬워서] | [93.95]% | [2] | [2] |

---
## 직접 해보기 (도전) - 게으름뱅이를 반대로 만들면

- 상황: 전부 양품이라 답하는 모델을 만들어봤다. 반대는 어떨까
- 할 일: 전부 불량이라 답하는 모델의 점수를 재고, 추천 모델을 하나 더 붙여 표를 늘린다
- 결과물: 네 줄짜리 기록표 1개

In [5]:
# numpy - 숫자 묶음을 다루는 도구를 np라는 짧은 이름으로 불러온다
import numpy as np

# 시험용 개수만큼 전부 0(양품)으로 채운 답안지를 만든다. 학습은 하지 않았다
기준예측 = np.ones(len(y_test), dtype=int)

# 맞힌 개수 ÷ 전체 개수
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델이 불량이라 한 건수:", 기준예측.sum())
print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")

기준 모델이 불량이라 한 건수: 314
기준 모델 정확도: 6.69 %


In [6]:
from sklearn.tree import DecisionTreeClassifier

# 1. 전부 양품이라 답하는 기준 모델 (새로 계산, 위에서 덮어쓴 변수는 건드리지 않음)
전부양품_예측 = np.zeros(len(y_test), dtype=int)
전부양품_정확도 = (전부양품_예측 == y_test).mean() * 100
전부양품_불량건수 = int((전부양품_예측 == 1).sum())
전부양품_진짜불량 = int(((전부양품_예측 == 1) & (y_test == 1)).sum())

# 2. 전부 불량이라 답하는 모델
전부불량_예측 = np.ones(len(y_test), dtype=int)
전부불량_정확도 = (전부불량_예측 == y_test).mean() * 100
전부불량_불량건수 = int((전부불량_예측 == 1).sum())
전부불량_진짜불량 = int(((전부불량_예측 == 1) & (y_test == 1)).sum())

# 4. 아직 안 써본 추천 모델: 의사결정나무 (단위를 맞출 필요 없음, 불균형 보정 없이 기본 상태)
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)
예측_트리 = tree_model.predict(X_test)

정확도_트리 = accuracy_score(y_test, 예측_트리) * 100
불량_예측_건수_트리 = int((예측_트리 == 1).sum())
그중_실제_불량_건수_트리 = int(((예측_트리 == 1) & (y_test == 1)).sum())

# 3번(로지스틱 회귀)은 앞서 계산한 정확도 / 불량_예측_건수 / 그중_실제_불량_건수 변수를 그대로 재사용
비교표_4모델 = pd.DataFrame({
    "모델": ["전부 양품이라 답하는 기준 모델", "전부 불량이라 답하는 모델", "로지스틱 회귀 (앞서 학습)", "결정트리 (방금 학습)"],
    "정확도(%)": [round(전부양품_정확도, 2), round(전부불량_정확도, 2), round(정확도, 2), round(정확도_트리, 2)],
    "불량이라 한 건수": [전부양품_불량건수, 전부불량_불량건수, 불량_예측_건수, 불량_예측_건수_트리],
    "그중 진짜 불량 건수": [전부양품_진짜불량, 전부불량_진짜불량, 그중_실제_불량_건수, 그중_실제_불량_건수_트리],
})
비교표_4모델


,모델,정확도(%),불량이라 한 건수,그중 진짜 불량 건수
0,전부 양품이라 답하는 기준 모델,93.31,0,0
1,전부 불량이라 답하는 모델,6.69,314,21
2,로지스틱 회귀 (앞서 학습),93.95,2,2
3,결정트리 (방금 학습),86.62,31,5


### 네 모델 비교

| 모델 | 정확도 | 불량이라 한 건수 | 그중 진짜 |
|---|---|---|---|
| 전부 양품 | [93.31]% | [0] | [0] |
| 전부 불량 | [6.69]% | [314] | [21] |
| [로지스틱 회귀] | [93.95]% | [2] | [2] |
| [의사결정나무] | [86.62]% | [31] | [5] |